# Controlled Decoding: шаг 3, оценка на test

Сравниваем на 500 test-промптах HH:

| Метод | Как выбирается ответ |
|---|---|
| π_ref | один сэмпл базовой модели (best-of-1) |
| best-of-K (RM) | K полных сэмплов, берём лучший по Skywork RM (бейзлайн статьи) |
| best-of-K (scorer) | те же K сэмплов, берём лучший по V(x, y) нашего scorer'а: проверка, насколько scorer заменяет RM |
| блочный CD-FUDGE (M, K) | K продолжений по M токенов, берём лучшее по V(x, yᵗ), повторяем до EOS или 256 токенов |

**Метрики** (все считаются по Skywork RM, той же модели, на которой учился scorer; в статье судья был отдельный, PaLM 2-L):
- средний reward с 95% ДИ по промптам;
- win rate против π_ref: доля сравнений с независимыми сэмплами базовой модели, где ответ метода получил больший reward;
- KL к π_ref, оценка сверху как в статье: log K − (K−1)/K для best-of-K. Для блочного CD это значение умножается на число блоков:
  на каждом блоке делается best-of-K по V;
- число сгенерированных токенов на промпт: вычислительная стоимость метода.

**Главный вопрос:** достигает ли блочный CD с малым K того же reward, что best-of-K с большим K, и какой ценой по KL и по токенам.

**Входы (Add Input):** датасеты `cd-hh-rollouts` (`test_prompts.parquet`, `train.parquet`, `config.json`) и `cd-hh-scorer`
(`scorer_config.json` и веса).

**Устройства:** π_ref на `cuda:0`, scorer и RM на `cuda:1`.

**Промежуточные сохранения:** каждая конфигурация считается шардами по `SHARD_SIZE` промптов. Шард, в котором уже есть
сгенерированные ответы и их reward, сохраняется в `cd_eval/shards`. Restart и Run All пропустят готовые шарды. Для новой сессии:
подключи датасет, который загружает последняя ячейка (`upload_dir` можно вызвать и вручную посреди прогона), и готовые шарды
подтянутся из `/kaggle/input`.

In [ ]:
import os, re, json, glob, time, shutil, math
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForTokenClassification, AutoModelForSequenceClassification

print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
DEBUG = True  # 16 промптов; для полного прогона поставить False

N_PROMPTS = 16 if DEBUG else 500
BON_K_MAX = 32                    # сэмплов π_ref на промпт для best-of-K
BON_KS = [1, 2, 4, 8, 16, 32]
N_REF = 4                         # ещё столько независимых сэмплов π_ref на промпт: опорные ответы для win rate
CD_CONFIGS = [(M, K) for K in [2, 4, 8] for M in [64, 32, 16]]  # (длина блока, число кандидатов), от дешёвых к дорогим
SHARD_SIZE = 8 if DEBUG else 50
GEN_SEQS = 128                    # последовательностей в одном вызове generate
SCORER_BATCH = 32
RM_BATCH = 32
SEED = 0

GEN_DEVICE = "cuda:0"
AUX_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
DTYPE = torch.float16

WORK = "/kaggle/working" if os.path.exists("/kaggle") else "."
OUT = f"{WORK}/cd_eval_debug" if DEBUG else f"{WORK}/cd_eval"
SHARDS = f"{OUT}/shards"
os.makedirs(SHARDS, exist_ok=True)

## Входы и защита от чужих шардов

In [ ]:
def find_one(pattern, what):
    found = [f for f in sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)) + sorted(glob.glob(pattern))
             if "debug" not in f]
    assert len(found) == 1, f"{what}: нужен ровно один {pattern}, найдено {found}"
    return os.path.dirname(found[0])

ROLLOUTS = find_one("test_prompts.parquet", "роллауты")
SCORER = find_one("scorer_config.json", "scorer")
rollout_cfg = json.load(open(f"{ROLLOUTS}/config.json"))
scorer_cfg = json.load(open(f"{SCORER}/scorer_config.json"))
print("rollouts:", ROLLOUTS, "\nscorer:", SCORER, "\nscorer val:", scorer_cfg["val"])
assert not rollout_cfg["debug"] and not scorer_cfg["train_meta"]["debug"], "подключены отладочные артефакты"
assert scorer_cfg["rollouts_cfg"] == rollout_cfg, "scorer обучен на других роллаутах"

GEN_KW = {k: v for k, v in rollout_cfg["gen_kw"].items() if k != "max_new_tokens"}  # тот же π_ref, что при обучении
MAX_NEW_TOKENS = rollout_cfg["gen_kw"]["max_new_tokens"]
print("GEN_KW:", GEN_KW, "MAX_NEW_TOKENS:", MAX_NEW_TOKENS)

test = pd.read_parquet(f"{ROLLOUTS}/test_prompts.parquet").head(N_PROMPTS)
test["prompt_ids"] = test.prompt_ids.map(lambda x: x.tolist())  # из parquet приходит numpy-массив; tok.pad принимает только int из Python
n_shards = (len(test) + SHARD_SIZE - 1) // SHARD_SIZE

RUN_META = {"rollouts_cfg": rollout_cfg, "scorer_train_meta": scorer_cfg["train_meta"], "n_prompts": N_PROMPTS,
            "bon_k_max": BON_K_MAX, "n_ref": N_REF, "shard_size": SHARD_SIZE, "seed": SEED, "rm": rollout_cfg["rm_model"]}
meta_path = f"{SHARDS}/run_meta.json"
if os.path.exists(meta_path):
    assert json.load(open(meta_path)) == RUN_META, f"В {SHARDS} шарды от прогона с другими параметрами. Удали папку: !rm -rf {SHARDS}"
elif glob.glob(f"{SHARDS}/*.parquet"):
    raise RuntimeError(f"В {SHARDS} шарды неизвестного прогона. Удали папку: !rm -rf {SHARDS}")
else:
    json.dump(RUN_META, open(meta_path, "w"), indent=2)

# Шарды из прошлой сессии (датасет, загруженный ячейкой upload_dir), если параметры совпадают
for prev_meta in glob.glob("/kaggle/input/**/run_meta.json", recursive=True):
    if json.load(open(prev_meta)) == RUN_META:
        n = 0
        for f in glob.glob(os.path.join(os.path.dirname(prev_meta), "*.parquet")):
            if not os.path.exists(f"{SHARDS}/{os.path.basename(f)}"):
                shutil.copy(f, SHARDS); n += 1
        print(f"подтянул {n} шардов из {os.path.dirname(prev_meta)}")

## Модели

In [ ]:
tok = AutoTokenizer.from_pretrained(rollout_cfg["base_model"])
tok.padding_side = "left"
policy = AutoModelForCausalLM.from_pretrained(rollout_cfg["base_model"], dtype=DTYPE).to(GEN_DEVICE).eval()
eos = policy.generation_config.eos_token_id
EOS_IDS = set(eos if isinstance(eos, list) else [eos])

scorer = AutoModelForTokenClassification.from_pretrained(SCORER, dtype=DTYPE).to(AUX_DEVICE).eval()
scorer.config.pad_token_id = tok.pad_token_id

rm_tok = AutoTokenizer.from_pretrained(rollout_cfg["rm_model"])
rm_tok.padding_side = "right"
rm = AutoModelForSequenceClassification.from_pretrained(rollout_cfg["rm_model"], dtype=DTYPE, num_labels=1).to(AUX_DEVICE).eval()
rm.config.pad_token_id = rm_tok.pad_token_id

def cut_at_eos(ids):
    for i, t in enumerate(ids):
        if t in EOS_IDS:
            return ids[: i + 1], True
    return ids, False

@torch.no_grad()
def sample(seqs, n, max_new):
    # n сэмплов π_ref на каждую последовательность; ответ i*n + k относится к seqs[i]
    out = []
    per_call = max(1, GEN_SEQS // n)
    for b in range(0, len(seqs), per_call):
        batch = tok.pad({"input_ids": seqs[b:b + per_call]}, return_tensors="pt").to(GEN_DEVICE)
        g = policy.generate(**batch, **GEN_KW, max_new_tokens=max_new, num_return_sequences=n, pad_token_id=tok.pad_token_id)
        out += [cut_at_eos(o) for o in g[:, batch["input_ids"].shape[1]:].tolist()]
    return out

@torch.no_grad()
def values(seqs):
    # V(x, y^t) на последнем токене каждой последовательности (промпт + префикс ответа)
    res = np.empty(len(seqs))
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    for k in range(0, len(order), SCORER_BATCH):
        idx = order[k:k + SCORER_BATCH]
        L = max(len(seqs[i]) for i in idx)
        ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            ids[r, :len(seqs[i])] = torch.tensor(seqs[i]); att[r, :len(seqs[i])] = 1
        v = scorer(input_ids=ids.to(AUX_DEVICE), attention_mask=att.to(AUX_DEVICE)).logits[..., 0].float()
        last = att.sum(1).to(AUX_DEVICE) - 1
        res[idx] = v[torch.arange(len(idx), device=AUX_DEVICE), last].cpu().numpy()
    return res

def rm_text(messages, response):
    t = rm_tok.apply_chat_template(json.loads(messages) + [{"role": "assistant", "content": response}], tokenize=False)
    return t[len(rm_tok.bos_token):] if rm_tok.bos_token and t.startswith(rm_tok.bos_token) else t

@torch.no_grad()
def rewards(messages, responses):
    texts = [rm_text(m, r) for m, r in zip(messages, responses)]
    res = []
    for b in range(0, len(texts), RM_BATCH):
        enc = rm_tok(texts[b:b + RM_BATCH], return_tensors="pt", padding=True).to(AUX_DEVICE)
        res += rm(**enc).logits[:, 0].float().tolist()
    res = np.array(res)
    assert np.isfinite(res).all()
    return res

def decode(ids):
    return tok.decode(ids, skip_special_tokens=True).strip()

## Проверка scorer'а после загрузки

Веса сохранены в fp16, а метрики в `02` считались до конвертации. Пересчитываем `pair_acc` на полном ответе на той же
валидации (тот же seed и VAL_FRAC=0.05, что в `02`) и сверяем с `scorer_config.json`. Заодно это проверяет соглашение о позициях.

In [ ]:
tr = pd.read_parquet(f"{ROLLOUTS}/train.parquet")
pids = np.random.default_rng(scorer_cfg["train_meta"]["seed"]).permutation(tr.pid.unique())
val = tr[tr.pid.isin(set(pids[: int(len(pids) * 0.05)].tolist()))].reset_index(drop=True)
assert len(val) == scorer_cfg["train_meta"]["n_val"]
val["v"] = values([list(p) + list(r) for p, r in zip(val.prompt_ids, val.response_ids)])
acc = []
for _, g in val.groupby("pid"):
    if len(g) == 2 and abs(g.reward.iloc[0] - g.reward.iloc[1]) > 0.5:
        acc.append((g.v.iloc[0] > g.v.iloc[1]) == (g.reward.iloc[0] > g.reward.iloc[1]))
print(f"pair_acc@full: сейчас {np.mean(acc):.3f} на {len(acc)} парах, в 02 было {scorer_cfg['val']['pair_acc@256']:.3f}")
assert abs(np.mean(acc) - scorer_cfg["val"]["pair_acc@256"]) < 0.03, "scorer после загрузки ведёт себя иначе, чем при обучении"
del tr

## Best-of-K и опорные сэмплы π_ref

На каждый промпт `BON_K_MAX + N_REF` сэмплов. Первые 32 идут на best-of-K, последние `N_REF` служат опорой для win rate.
Для каждого сэмпла считаем reward RM и V(x, y) scorer'а на полном ответе.

In [ ]:
def shard_path(name, s):
    return f"{SHARDS}/{name}_{s:03d}.parquet"

pbar = tqdm(total=len(test), desc="best-of-K", unit="prompt")
for s in range(n_shards):
    sh = test.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
    if os.path.exists(shard_path("bon", s)):
        pbar.update(len(sh)); continue
    t0 = time.time()
    torch.manual_seed(SEED * 100_003 + s)
    n = BON_K_MAX + N_REF
    res = sample(sh.prompt_ids.tolist(), n, MAX_NEW_TOKENS)
    rows = []
    for j, p in enumerate(sh.itertuples()):
        for k in range(n):
            ids, fin = res[j * n + k]
            rows.append({"pid": p.pid, "sample": k, "messages": p.messages, "seq": p.prompt_ids + ids,
                         "response_ids": ids, "finished": fin, "response": decode(ids)})
    df = pd.DataFrame(rows)
    df["reward"] = rewards(df.messages, df.response)
    df["v"] = values(df.seq.tolist())
    df.drop(columns=["seq", "messages"]).to_parquet(shard_path("bon", s))
    pbar.update(len(sh))
    pbar.write(f"bon shard {s + 1}/{n_shards}: {time.time() - t0:.0f}s")
pbar.close()

## Блочный CD-FUDGE

На каждом шаге для всех ещё не законченных промптов генерируем K продолжений по M токенов (продолжение может закончиться
EOS раньше), считаем V(x, префикс + блок) и оставляем лучшее. Промпт закончен, когда выбранный блок закончился EOS или
ответ дошёл до 256 токенов.

In [ ]:
def run_cd(prompt_ids, M, K):
    n = len(prompt_ids)
    resp, fin, n_blocks, gen_tok = [[] for _ in range(n)], [False] * n, [0] * n, [0] * n
    done = [False] * n
    while not all(done):
        act = [i for i in range(n) if not done[i]]
        seqs = [prompt_ids[i] + resp[i] for i in act]
        max_new = min(M, max(MAX_NEW_TOKENS - len(resp[i]) for i in act))
        cands = sample(seqs, K, max_new)
        flat, meta = [], []
        for j, i in enumerate(act):
            rem = MAX_NEW_TOKENS - len(resp[i])
            for k in range(K):
                ids, f = cands[j * K + k]
                if len(ids) > rem:
                    ids, f = ids[:rem], False
                gen_tok[i] += len(ids)
                flat.append(seqs[j] + ids); meta.append((ids, f))
        v = values(flat)
        for j, i in enumerate(act):
            ids, f = meta[j * K + int(np.argmax(v[j * K:(j + 1) * K]))]
            resp[i] += ids; n_blocks[i] += 1; fin[i] = f
            done[i] = f or len(resp[i]) >= MAX_NEW_TOKENS
    return resp, fin, n_blocks, gen_tok

for ci, (M, K) in enumerate(CD_CONFIGS):
    name = f"cd_M{M}_K{K}"
    pbar = tqdm(total=len(test), desc=name, unit="prompt")
    t_cfg = time.time()
    for s in range(n_shards):
        sh = test.iloc[s * SHARD_SIZE:(s + 1) * SHARD_SIZE]
        if os.path.exists(shard_path(name, s)):
            pbar.update(len(sh)); continue
        torch.manual_seed(SEED * 100_003 + 1000 * (ci + 1) + s)
        resp, fin, nb, gt = run_cd(sh.prompt_ids.tolist(), M, K)
        df = pd.DataFrame({"pid": sh.pid.values, "response_ids": resp, "finished": fin, "n_blocks": nb, "gen_tokens": gt,
                           "response": [decode(r) for r in resp]})
        df["reward"] = rewards(sh.messages.tolist(), df.response.tolist())
        df.to_parquet(shard_path(name, s))
        pbar.update(len(sh))
    pbar.close()
    print(f"{name}: {time.time() - t_cfg:.0f}s")

## Сводка

In [ ]:
def load(name):
    return pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f"{SHARDS}/{name}_*.parquet"))], ignore_index=True)

bon = load("bon")
assert bon.pid.nunique() == len(test)
ref = {pid: g.reward.values for pid, g in bon[bon["sample"] >= BON_K_MAX].groupby("pid")}  # опорные ответы π_ref

def win_rate(pid, r):
    rr = ref[pid]
    return np.mean((r > rr) + 0.5 * (r == rr))

def kl_bon(K):
    return math.log(K) - (K - 1) / K

def summarize(per_prompt, method, K, M, kl, tokens):
    # per_prompt: DataFrame с колонками pid, reward, win (среднее по повторам внутри промпта, если они есть)
    r, w = per_prompt.reward.values, per_prompt.win.values
    return {"method": method, "K": K, "M": M, "kl_bound": kl, "gen_tokens": tokens,
            "reward": r.mean(), "reward_ci": 1.96 * r.std(ddof=1) / np.sqrt(len(r)),
            "win_rate": w.mean(), "win_ci": 1.96 * w.std(ddof=1) / np.sqrt(len(w))}

rows = []
pool = bon[bon["sample"] < BON_K_MAX]
for K in BON_KS:
    # 32 сэмпла режем на 32/K непересекающихся групп и усредняем: так best-of-K для малых K менее шумный
    for by in ["rm", "scorer"]:
        recs = []
        for pid, g in pool.groupby("pid"):
            g = g.sort_values("sample")
            r, v = g.reward.values.reshape(-1, K), g.v.values.reshape(-1, K)
            pick = r.argmax(1) if by == "rm" else v.argmax(1)
            chosen = r[np.arange(len(r)), pick]
            recs.append({"pid": pid, "reward": chosen.mean(), "win": np.mean([win_rate(pid, c) for c in chosen])})
        tokens = K * pool.response_ids.map(len).mean()
        method = "π_ref" if K == 1 else f"best-of-K ({by})"
        if K == 1 and by == "scorer":
            continue
        rows.append(summarize(pd.DataFrame(recs), method, K, None, kl_bon(K), tokens))

for M, K in CD_CONFIGS:
    name = f"cd_M{M}_K{K}"
    if not glob.glob(f"{SHARDS}/{name}_*.parquet"):
        continue
    cd = load(name)
    cd["win"] = [win_rate(p, r) for p, r in zip(cd.pid, cd.reward)]
    rows.append(summarize(cd, "blockwise CD", K, M, (cd.n_blocks * kl_bon(K)).mean(), cd.gen_tokens.mean())
                | {"finished": cd.finished.mean(), "resp_len": cd.response_ids.map(len).mean(), "n_prompts": len(cd)})

summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/summary.csv", index=False)
pd.set_option("display.width", 200)
print(summary.round(3).to_string(index=False))

## Графики

- Слева reward против KL: кривая лучше, если она выше и левее. По этим осям строятся главные графики статьи.
- В центре win rate против π_ref по тем же осям.
- Справа reward против числа сгенерированных токенов на промпт. Это вопрос «дешевле ли CD, чем best-of-K».

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
bo = summary[summary.method.isin(["π_ref", "best-of-K (rm)"])].sort_values("K")
bs = summary[summary.method.isin(["π_ref", "best-of-K (scorer)"])].sort_values("K")
for a, (x, xl) in zip(ax, [("kl_bound", "KL к π_ref (оценка сверху)"), ("kl_bound", "KL к π_ref (оценка сверху)"), ("gen_tokens", "сгенерировано токенов на промпт")]):
    y, yerr = ("win_rate", "win_ci") if a is ax[1] else ("reward", "reward_ci")
    a.errorbar(bo[x], bo[y], yerr=bo[yerr], fmt="o-", color="black", label="best-of-K (RM)", capsize=2)
    a.errorbar(bs[x], bs[y], yerr=bs[yerr], fmt="s--", color="gray", label="best-of-K (scorer)", capsize=2)
    for M, c in zip([16, 32, 64], ["tab:blue", "tab:orange", "tab:green"]):
        d = summary[(summary.method == "blockwise CD") & (summary.M == M)].sort_values("K")
        if len(d):
            a.errorbar(d[x], d[y], yerr=d[yerr], fmt="^-", color=c, label=f"CD, M={M}", capsize=2)
            for _, r in d.iterrows():
                a.annotate(f"K={r.K}", (r[x], r[y]), fontsize=7, xytext=(3, -9), textcoords="offset points")
    a.set_xlabel(xl); a.set_ylabel(y); a.grid(alpha=0.3)
ax[2].set_xscale("log", base=2)
ax[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/curves.png", dpi=150); plt.show()

## Примеры

Тот же промпт: π_ref, best-of-32 по RM и CD с самой дорогой конфигурацией. Глазами проверяем, не взламывает ли CD reward:
не выдаёт ли он, например, шаблонные отказы, которые RM любит больше полезных ответов.

In [ ]:
last_cd = next((f"cd_M{M}_K{K}" for M, K in reversed(CD_CONFIGS) if glob.glob(f"{SHARDS}/cd_M{M}_K{K}_*.parquet")), None)
cdx = load(last_cd).set_index("pid") if last_cd else None
for pid in test.pid.sample(3, random_state=SEED):
    g = pool[pool.pid == pid].sort_values("sample")
    print("=" * 100, "\nUSER:", json.loads(test.set_index("pid").messages[pid])[-1]["content"][:400])
    print(f"\n--- π_ref (reward {g.reward.iloc[0]:.2f}):\n{g.response.iloc[0][:600]}")
    b = g.loc[g.reward.idxmax()]
    print(f"\n--- best-of-{BON_K_MAX} RM (reward {b.reward:.2f}):\n{b.response[:600]}")
    if cdx is not None:
        print(f"\n--- {last_cd} (reward {cdx.reward[pid]:.2f}):\n{cdx.response[pid][:600]}")

## Парные сравнения

Все методы оцениваются на одних и тех же промптах, а reward сильно зависит от самого промпта (corr@0 ≈ 0,5 в `02`).
Поэтому сравниваем методы попарно на одном промпте и бутстрепим по промптам. Так ДИ разницы получается намного уже, чем
разница двух отдельных ДИ.

Для каждой конфигурации CD:
- **K_eq**: при каком K best-of-K (RM) даёт тот же средний reward. Считается интерполяцией по log K, с 95% ДИ.
 Это аналог фразы из статьи «CD с K=6 соответствует best-of-50». `≥32` значит, что CD лучше всей кривой best-of-K.
- **Δ при равных токенах**: CD минус best-of-K с наименьшим K, которому досталось не меньше токенов, чем CD.
 Сравнение консервативное: best-of-K получает не меньший бюджет.
- **Δ при равном KL**: CD минус best-of-K с наименьшим K, у которого оценка KL не меньше, чем у CD. Если KL у CD больше,
 чем у best-of-32, берётся best-of-32 и ставится пометка `>`.
- `P(Δ≤0)`: доля бутстреп-выборок, где CD не лучше. Чем ближе к нулю, тем увереннее CD лучше.

Ячейка не использует GPU и модели. В новой сессии её можно выполнить отдельно, вместе с ячейкой импортов: подключи
датасет `cd-hh-eval`, шарды найдутся сами.

In [ ]:
N_BOOT = 5000

src_cands = ([SHARDS] if "SHARDS" in globals() else []) + sorted(os.path.dirname(f) for f in glob.glob("/kaggle/input/**/run_meta.json", recursive=True))
SRC = next(d for d in src_cands if glob.glob(f"{d}/bon_*.parquet"))
meta = json.load(open(f"{SRC}/run_meta.json"))
kmax = meta["bon_k_max"]
print("шарды:", SRC, "| промптов:", meta["n_prompts"])

def load_src(name):
    return pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{SRC}/{name}_*.parquet"))], ignore_index=True)

pool = load_src("bon")
pool = pool[pool["sample"] < kmax].sort_values(["pid", "sample"])
pids = np.sort(pool.pid.unique())
n = len(pids)
R = pool.reward.values.reshape(n, kmax)
Ks = [2 ** i for i in range(int(math.log2(kmax)) + 1)]
BON = np.stack([R.reshape(n, -1, K).max(2).mean(1) for K in Ks], 1)  # (промпт, K): как в сводке, по непересекающимся группам
bon_tok = np.array(Ks) * pool.response_ids.map(len).mean()
bon_kl = np.array([math.log(K) - (K - 1) / K for K in Ks])

rng = np.random.default_rng(0)
idx = rng.integers(0, n, (N_BOOT, n))
bon_boot = BON[idx].mean(1)  # (бутстреп, K)

def k_eq(c, curve):
    # K (интерполяция по log2 K), при котором кривая best-of-K доходит до reward c; кривую делаем монотонной
    curve = np.maximum.accumulate(curve)
    return 2 ** np.interp(c, curve, np.log2(Ks))

def ci(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return f"[{lo:.2f}; {hi:.2f}]"

rows = []
for f in sorted(set(re.sub(r"_\d+\.parquet$", "", os.path.basename(f)) for f in glob.glob(f"{SRC}/cd_M*_K*_*.parquet"))):
    M, K = map(int, re.match(r"cd_M(\d+)_K(\d+)", f).groups())
    cd = load_src(f).sort_values("pid")
    assert (cd.pid.values == pids).all(), f"{f}: не все промпты посчитаны"
    c = cd.reward.values
    c_boot = c[idx].mean(1)
    tok, kl = cd.gen_tokens.mean(), (cd.n_blocks * (math.log(K) - (K - 1) / K)).mean()
    ke = k_eq(c.mean(), BON.mean(0))
    ke_boot = np.array([k_eq(cb, bb) for cb, bb in zip(c_boot, bon_boot)])
    row = {"M": M, "K": K, "tokens": round(tok), "kl": round(kl, 2), "reward": round(c.mean(), 2),
           "K_eq": ("≥" if ke >= kmax else "") + f"{ke:.1f}", "K_eq_ci": ci(ke_boot)}
    for label, j in [("tok", int(np.searchsorted(bon_tok, tok))), ("kl", int(np.searchsorted(bon_kl, kl)))]:
        over = j >= len(Ks)
        j = min(j, len(Ks) - 1)
        d = c[idx].mean(1) - BON[idx, j].mean(1)
        row[f"vs_{label}"] = (">" if over else "") + f"best-of-{Ks[j]}"
        row[f"Δ_{label}"] = round(c.mean() - BON[:, j].mean(), 2)
        row[f"Δ_{label}_ci"] = ci(d)
        row[f"P(Δ_{label}≤0)"] = round(float((d <= 0).mean()), 3)
    rows.append(row)

paired = pd.DataFrame(rows).sort_values(["K", "M"])
paired.to_csv(f"{OUT if 'OUT' in globals() else '.'}/paired.csv", index=False)
print("\nbest-of-K (RM):", ", ".join(f"K={K}: {r:.2f}" for K, r in zip(Ks, BON.mean(0))))
print(paired.to_string(index=False))

## Сохранение результатов в Kaggle Dataset

Загружает шарды, `summary.csv` и `curves.png` в приватный датасет: при первом запуске создаёт его, потом добавляет версии.
Посреди прогона можно вызвать вручную `upload_dir(SHARDS, SLUG)`. Тогда в новой сессии подключи этот датасет, и готовые
шарды подтянутся. Нужен секрет `KAGGLE_API_TOKEN`.

In [ ]:
!pip install -q -U kaggle

import subprocess
from kaggle_secrets import UserSecretsClient
os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")

USERNAME = "karimvafin"
SLUG = "cd-hh-eval-debug" if DEBUG else "cd-hh-eval"

def kaggle_cli(*args):
    r = subprocess.run(["kaggle", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).strip()

def upload_dir(src, slug, extra_files=(), message="update"):
    # Kaggle CLI без --dir-mode пропускает подпапки, поэтому всё кладём в один плоский каталог
    stage = f"{WORK}/upload_{slug}"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(src, stage)
    for f in extra_files:
        shutil.copy(f, stage)
    ds_id = f"{USERNAME}/{slug}"
    json.dump({"title": slug, "id": ds_id, "licenses": [{"name": "CC0-1.0"}]}, open(f"{stage}/dataset-metadata.json", "w"))
    exists = kaggle_cli("datasets", "status", ds_id)[0] == 0
    code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message) if exists else kaggle_cli("datasets", "create", "-p", stage)
    if code != 0 and not exists:
        # status мог ошибочно сказать, что датасета нет (например, он остался от прошлой попытки): пробуем как новую версию
        print(out[-2000:], "\n→ create не удался, пробую version")
        code, out = kaggle_cli("datasets", "version", "-p", stage, "-m", message)
        exists = True
    print(out[-2000:])
    assert code == 0, "загрузка не удалась, см. вывод выше; файлы остались в " + stage
    shutil.rmtree(stage, ignore_errors=True)
    print(f"{'новая версия' if exists else 'создан'}: https://www.kaggle.com/datasets/{ds_id}")

upload_dir(SHARDS, SLUG, extra_files=[f"{OUT}/summary.csv", f"{OUT}/curves.png", f"{OUT}/paired.csv"], message=f"{len(summary)} rows in summary")